In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install mediapipe opencv-python numpy -q

import urllib.request
# 下載 Pose 和 Hand 模型（各約 5–30MB，只需下載一次）
urllib.request.urlretrieve(
    'https://storage.googleapis.com/mediapipe-models/pose_landmarker/pose_landmarker_lite/float16/1/pose_landmarker_lite.task',
    '/tmp/pose_landmarker.task'
)
urllib.request.urlretrieve(
    'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task',
    '/tmp/hand_landmarker.task'
)
print("模型下載完成")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 100.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.8/135.8 kB 13.1 MB/s eta 0:00:00
模型下載完成


In [ ]:
import os
import csv
import logging
import numpy as np
import cv2
from pathlib import Path
import mediapipe as mp
from mediapipe.tasks import python as mp_python
from mediapipe.tasks.python import vision as mp_vision

# ── 設定 ─────────────────────────────────────────────────────
BASE_DIR      = Path('/content/drive/MyDrive/4016project/27261843')
METADATA_CSV  = BASE_DIR / "splits" / "metadata.csv"
LANDMARKS_DIR = BASE_DIR / "landmarks"
LOG_FILE      = LANDMARKS_DIR / "extract_log.txt"

POSE_MODEL  = '/tmp/pose_landmarker.task'
HAND_MODEL  = '/tmp/hand_landmarker.task'
MP_CONFIDENCE = 0.5
D = 225  # 33×3 + 21×3 + 21×3

LEFT_SHOULDER  = 11
RIGHT_SHOULDER = 12
# ─────────────────────────────────────────────────────────────

LANDMARKS_DIR.mkdir(parents=True, exist_ok=True)
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(message)s",
    handlers=[
        logging.FileHandler(LOG_FILE, encoding="utf-8"),
        logging.StreamHandler()
    ]
)
log = logging.getLogger(__name__)

# ── 建立 detector ─────────────────────────────────────────────
pose_opts = mp_vision.PoseLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=POSE_MODEL),
    running_mode=mp_vision.RunningMode.IMAGE,
    min_pose_detection_confidence=MP_CONFIDENCE,
)
hand_opts = mp_vision.HandLandmarkerOptions(
    base_options=mp_python.BaseOptions(model_asset_path=HAND_MODEL),
    running_mode=mp_vision.RunningMode.IMAGE,
    num_hands=2,
    min_hand_detection_confidence=MP_CONFIDENCE,
)
pose_detector = mp_vision.PoseLandmarker.create_from_options(pose_opts)
hand_detector = mp_vision.HandLandmarker.create_from_options(hand_opts)

# ── 抽單幀 ────────────────────────────────────────────────────
def extract_frame_landmarks(frame_bgr):
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    mp_image  = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

    pose_result = pose_detector.detect(mp_image)
    if not pose_result.pose_landmarks:
        return None

    # Pose: 33 × 3
    pose = np.array([[lm.x, lm.y, lm.z]
                     for lm in pose_result.pose_landmarks[0]],
                    dtype=np.float32).flatten()  # (99,)

    # Hand: 左右各 21 × 3，預設全 0
    lh = np.zeros(63, dtype=np.float32)
    rh = np.zeros(63, dtype=np.float32)

    hand_result = hand_detector.detect(mp_image)
    if hand_result.hand_landmarks:
        for i, hand_lms in enumerate(hand_result.hand_landmarks):
            # handedness: "Left" / "Right"（MediaPipe 係鏡像，Left=右手）
            label = hand_result.handedness[i][0].category_name
            arr = np.array([[lm.x, lm.y, lm.z] for lm in hand_lms],
                           dtype=np.float32).flatten()
            if label == "Left":
                lh = arr
            else:
                rh = arr

    return np.concatenate([pose, lh, rh])  # (225,)


# ── 正規化 ────────────────────────────────────────────────────
def normalize_sequence(seq):
    normalized  = seq.copy()
    l_shoulder  = seq[:, LEFT_SHOULDER*3  : LEFT_SHOULDER*3+3]
    r_shoulder  = seq[:, RIGHT_SHOULDER*3 : RIGHT_SHOULDER*3+3]
    midpoint    = (l_shoulder + r_shoulder) / 2.0
    shoulder_w  = np.linalg.norm(l_shoulder - r_shoulder, axis=1, keepdims=True)
    shoulder_w  = np.where(shoulder_w < 1e-6, 1.0, shoulder_w)

    for start, n in [(0, 33), (99, 21), (162, 21)]:
        end  = start + n * 3
        part = normalized[:, start:end].reshape(-1, n, 3)
        part = (part - midpoint[:, np.newaxis, :]) / shoulder_w[:, np.newaxis, :]
        normalized[:, start:end] = part.reshape(-1, n*3)
    return normalized


# ── 插值 ─────────────────────────────────────────────────────
def interpolate_missing(frames_raw):
    T = len(frames_raw)
    valid = [i for i, f in enumerate(frames_raw) if f is not None]
    if not valid:
        return None
    result = np.zeros((T, D), dtype=np.float32)
    for i in range(T):
        if frames_raw[i] is not None:
            result[i] = frames_raw[i]
        else:
            prev_v = [v for v in valid if v < i]
            next_v = [v for v in valid if v > i]
            if prev_v and next_v:
                p, n  = prev_v[-1], next_v[0]
                alpha = (i - p) / (n - p)
                result[i] = (1-alpha)*frames_raw[p] + alpha*frames_raw[n]
            elif prev_v:
                result[i] = frames_raw[prev_v[-1]]
            else:
                result[i] = frames_raw[next_v[0]]
    return result


# ── 主流程 ────────────────────────────────────────────────────
def main():
    samples = []
    with open(METADATA_CSV, encoding="utf-8") as f:
        for row in csv.DictReader(f):
            samples.append(row)
    log.info(f"共 {len(samples)} 個樣本")

    stats = {"success": 0, "skipped": 0, "missing_frames": 0}

    for idx, row in enumerate(samples):
        sample_id = row["sample_id"]
        clip_path = BASE_DIR / row["path"]
        out_path  = LANDMARKS_DIR / f"{sample_id}.npy"

        if out_path.exists():
            stats["success"] += 1
            continue

        frame_files = sorted(clip_path.glob("*.jpg")) + sorted(clip_path.glob("*.jpeg"))
        if not frame_files:
            log.warning(f"[{idx+1}] NO FRAMES: {sample_id}")
            stats["skipped"] += 1
            continue

        frames_raw    = []
        missing_count = 0
        for fpath in frame_files:
            img = cv2.imread(str(fpath))
            if img is None:
                frames_raw.append(None)
                missing_count += 1
                continue
            lm = extract_frame_landmarks(img)
            if lm is None:
                missing_count += 1
            frames_raw.append(lm)

        seq = interpolate_missing(frames_raw)
        if seq is None:
            log.warning(f"[{idx+1}] DISCARD: {sample_id}")
            stats["skipped"] += 1
            continue

        stats["missing_frames"] += missing_count
        seq = normalize_sequence(seq)
        np.save(str(out_path), seq)
        stats["success"] += 1

        if (idx+1) % 100 == 0:
            log.info(f"進度: {idx+1}/{len(samples)} | 成功={stats['success']} 跳過={stats['skipped']}")

    log.info("=== 完成 ===")
    log.info(f"成功:{stats['success']} 跳過:{stats['skipped']} 插值幀:{stats['missing_frames']}")

In [ ]:
main()